In [4]:
print("RetailFlow Data Generation started")

RetailFlow Data Generation started


In [6]:
%pip install faker 

Note: you may need to restart the kernel to use updated packages.


In [7]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install numpy
%pip install matplotlib 
%pip install seaborn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [9]:
import uuid
import random
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta

# Set up Faker and fix all random seeds so results are repeatable
fake = Faker()
Faker.seed(101)
random.seed(101)
np.random.seed(101)

# 1. CUSTOMERS.CSV
print("Generating customers.csv...")

num_customers = 5000

cust_ids = [f"CUST_{10000 + i}" for i in range(num_customers)]
customer_names = [fake.name() for _ in range(num_customers)]
emails = [fake.email() if random.random() > 0.08 else np.nan for _ in range(num_customers)]
countries = [fake.country_code() for _ in range(num_customers)]

df_customers = pd.DataFrame({
    "customer_id": cust_ids,
    "customer_name": customer_names,
    "email": emails,
    "country": countries
})

# Add 150 duplicate rows
duplicate_rows = df_customers.sample(n=150, random_state=101).copy()
df_customers = pd.concat([df_customers, duplicate_rows], ignore_index=True)
df_customers.to_csv("customers.csv", index=False)

# 2. PRODUCTS.CSV
print("Generating products.csv...")

num_products = 800
categories = ['Electronics', 'Clothing', 'Home & Kitchen', 'Beauty', 'Sports']

prod_ids = [f"PROD_{1000 + i}" for i in range(num_products)]
product_names = [fake.catch_phrase() for _ in range(num_products)]
product_categories = [random.choice(categories) for _ in range(num_products)]
unit_prices = [round(random.uniform(4.99, 299.99), 2) for _ in range(num_products)]
active_flags = [random.choice([True, True, True, False]) for _ in range(num_products)]

df_products = pd.DataFrame({
    "product_id": prod_ids,
    "product_name": product_names,
    "category": product_categories,
    "unit_price": unit_prices,
    "active_flag": active_flags
})
df_products.to_csv("products.csv", index=False)

# 3. HELPER FUNCTION: build one batch of orders + order items
def generate_order_batch(num_orders, base_date, add_discount_code=False):
    orders, order_items = [], []
    regions = ['North', 'South', 'East', 'West', 'Central']
    statuses = ['Completed', 'Completed', 'Completed', 'Returned', 'Cancelled']

    for _ in range(num_orders):
        order_id = f"ORD_{uuid.uuid4().hex[:10].upper()}"
        cust_id = random.choice(cust_ids)
        order_ts = base_date + timedelta(
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )

        order_header = {
            "order_id": order_id,
            "customer_id": cust_id,
            "order_ts": order_ts.strftime('%Y-%m-%d %H:%M:%S'),
            "store_region": random.choice(regions),
            "status": random.choice(statuses)
        }

        discount_code = None
        if add_discount_code:
            discount_code = random.choice(['SAVE10', 'FREESHIP', None, None, 'WELCOME20'])
            order_header["discount_code"] = discount_code

        orders.append(order_header)

        num_items = random.choices([1, 2, 3, 4, 5], weights=[30, 25, 20, 15, 10])[0]
        chosen_products = df_products.sample(n=num_items)

        for _, product in chosen_products.iterrows():
            qty = random.randint(1, 4)
            unit_price = product['unit_price']
            line_total = round(qty * unit_price, 2)

            item = {
                "order_id": order_id,
                "product_id": product['product_id'],
                "quantity": qty,
                "unit_price": unit_price,
                "line_total": line_total
            }
            if add_discount_code:
                item["discount_code"] = discount_code

            order_items.append(item)

    return pd.DataFrame(orders), pd.DataFrame(order_items)

# 4. DAY 1 ORDERS & ORDER ITEMS
print("Generating Day 1 Orders Data...")
day1_date = datetime(2026, 7, 5)
df_orders_d1, df_items_d1 = generate_order_batch(20000, day1_date, add_discount_code=False)
df_orders_d1.to_json("orders_day1.json", orient="records", lines=True)
df_items_d1.to_json("order_items_day1.json", orient="records", lines=True)

# 5. DAY 2 ORDERS & ORDER ITEMS (adds the discount_code column)
print("Generating Day 2 Orders Data (With Schema Evolution)...")
day2_date = datetime(2026, 7, 6)
df_orders_d2, df_items_d2 = generate_order_batch(4000, day2_date, add_discount_code=True)
df_orders_d2.to_json("orders_day2.json", orient="records", lines=True)
df_items_d2.to_json("order_items_day2.json", orient="records", lines=True)

# 6. CLICKSTREAM LOGS
def generate_clickstream(num_events, base_date, filename):
    print(f"Generating clickstream events for {filename}...")
    events = ['view_item', 'add_to_cart', 'remove_from_cart', 'search', 'click_ad', 'checkout_start']
    clickstream = []

    for _ in range(num_events):
        event_ts = base_date + timedelta(
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59),
            seconds=random.randint(0, 59)
        )
        customer_id = random.choice(cust_ids) if random.random() > 0.3 else None

        clickstream.append({
            "session_id": str(uuid.uuid4()),
            "customer_id": customer_id,
            "event_timestamp": event_ts.strftime('%Y-%m-%d %H:%M:%S.%f')[:-3],
            "event_type": random.choice(events),
            "page_url": fake.uri_path(),
            "device_type": random.choice(['Desktop', 'Mobile', 'Mobile', 'Tablet'])
        })

    df_click = pd.DataFrame(clickstream).sort_values(by="event_timestamp")
    df_click.to_json(filename, orient="records", lines=True)

generate_clickstream(15000, day1_date, "clickstream_day1.json")
generate_clickstream(15000, day2_date, "clickstream_day2.json")

print("\nAll RetailFlow datasets generated.")


Generating customers.csv...
Generating products.csv...
Generating Day 1 Orders Data...
Generating Day 2 Orders Data (With Schema Evolution)...
Generating clickstream events for clickstream_day1.json...
Generating clickstream events for clickstream_day2.json...

All RetailFlow datasets generated.


In [10]:
import os
import pandas as pd

# Folder where your CSV/JSON files are located
output_dir = os.getcwd()

files_to_profile = [
    ("customers.csv", "csv"),
    ("products.csv", "csv"),
    ("orders_day1.json", "json"),
    ("orders_day2.json", "json"),
    ("order_items_day1.json", "json"),
    ("order_items_day2.json", "json"),
    ("clickstream_day1.json", "json"),
    ("clickstream_day2.json", "json"),
]

profile_registry = {}
null_heatmap_data = {}
report_lines = []

for fname, ftype in files_to_profile:
    path = os.path.join(output_dir, fname)

    # Load file depending on type
    if ftype == "csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_json(path, lines=True)

    profile_registry[fname] = df

    # Use extend instead of multiple append calls
    report_lines.extend([
        f"FILE: {fname}",
        f"Shape: {df.shape[0]} rows, {df.shape[1]} columns",
        f"Total Duplicates: {df.duplicated().sum()}"
    ])

    # Collect null percentages for heatmap
    for col in df.columns:
        null_percentage = df[col].isnull().mean() * 100
        null_heatmap_data[f"{fname} -> {col}"] = null_percentage

    # Column-level stats
    column_stats = []
    for col in df.columns:
        null_rate = df[col].isnull().mean() * 100
        dtype = str(df[col].dtype)

        non_nulls = df[col].dropna()
        if pd.api.types.is_numeric_dtype(df[col]) and not non_nulls.empty:
            value_range = f"[{non_nulls.min()} to {non_nulls.max()}]"
        elif not non_nulls.empty:
            value_range = f"[{non_nulls.nunique()} distinct clean values]"
        else:
            value_range = "[All values Null]"

        column_stats.append({
            "Column": col,
            "Dtype": dtype,
            "Null %": f"{null_rate:.2f}%",
            "Range/Cardinality": value_range
        })

    stats_df = pd.DataFrame(column_stats)
    report_lines.append(stats_df.to_string(index=False))
    report_lines.append("-" * 60)

# Build final report text
report_text = "\n".join(report_lines)

# Save to Markdown file
report_path = os.path.join(output_dir, "data_profile_report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print(f"Report saved to: {report_path}")


Report saved to: c:\Users\ASUS\AppData\Local\Programs\Microsoft VS Code\data_profile_report.md


In [11]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set chart style globally
sns.set_theme(style="whitegrid")

# Ensure charts directory exists
charts_dir = os.path.join(os.getcwd(), "charts")
os.makedirs(charts_dir, exist_ok=True)

# --- CHART 1: Order Volume by Day ---
days = ["Day 1 (July 1)", "Day 2 (July 2)"]
volumes = [
    len(profile_registry["orders_day1.json"]),
    len(profile_registry["orders_day2.json"])
]

fig1, ax1 = plt.subplots(figsize=(6, 4))
sns.barplot(x=days, y=volumes, palette="Blues_d", ax=ax1)
ax1.set_title("Order Batch Volume Comparison (Ingestion Pipeline)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Total Recorded Header Rows")

# Annotate bar values
for i, v in enumerate(volumes):
    ax1.text(i, v + 300, f"{v:,}", ha="center", weight="bold")

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "order_volume_by_day.png"), dpi=150)
plt.close()

# --- CHART 2: Revenue Distribution ---
df_items_day1 = profile_registry["order_items_day1.json"]
df_items_day2 = profile_registry["order_items_day2.json"]
df_items = pd.concat([df_items_day1, df_items_day2])

fig2, ax2 = plt.subplots(figsize=(7, 4))
sns.histplot(df_items["line_total"], bins=50, kde=True, color="teal", ax=ax2)
ax2.set_title("Distribution of Line Item Revenue Outliers", fontsize=12, fontweight="bold")
ax2.set_xlabel("Line Item Gross Totals ($)")
ax2.set_xlim(0, 300)  # cut the long tail for clarity

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "revenue_distribution.png"), dpi=150)
plt.close()

# --- CHART 3: Top Categories by Revenue ---
df_products = profile_registry["products.csv"]
df_all_items = pd.merge(df_items, df_products, on="product_id", how="left")

category_revenue = (
    df_all_items.groupby("category")["line_total"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig3, ax3 = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=category_revenue, x="line_total", y="category", palette="viridis", ax=ax3)
ax3.set_title("Top Product Categories by Total Gross Revenue", fontsize=12, fontweight="bold")
ax3.set_xlabel("Aggregated Total Revenue ($)")
ax3.set_ylabel("Category")
ax3.ticklabel_format(style="plain", axis="x")  # show plain numbers, not scientific notation

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "top_categories_revenue.png"), dpi=150)
plt.close()

# --- CHART 4: Null-Rate Heatmap Across All Files ---
heatmap_rows = [
    {"File": f.split(" -> ")[0], "Column": f.split(" -> ")[1], "Null_Percentage": p}
    for f, p in null_heatmap_data.items()
]

heatmap_df = pd.DataFrame(heatmap_rows)
pivot_heatmap = heatmap_df.pivot(index="Column", columns="File", values="Null_Percentage").fillna(0)

fig4, ax4 = plt.subplots(figsize=(10, 6))
sns.heatmap(
    pivot_heatmap,
    annot=True,
    fmt=".1f",
    cmap="Reds",
    cbar_kws={"label": "Null Rate (%)"},
    ax=ax4
)
ax4.set_title("Global Data Quality Heatmap: Missing Value Concentrations", fontsize=12, fontweight="bold")
plt.xticks(rotation=30, ha="right")

plt.tight_layout()
plt.savefig(os.path.join(charts_dir, "null_rate_heatmap.png"), dpi=150)
plt.close()

print(f"All 4 plots successfully compiled and exported to '{charts_dir}/'.")


C:\Users\ASUS\AppData\Local\Temp\ipykernel_8880\1509337641.py:21: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=days, y=volumes, palette="Blues_d", ax=ax1)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_8880\1509337641.py:60: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_revenue, x="line_total", y="category", palette="viridis", ax=ax3)


All 4 plots successfully compiled and exported to 'c:\Users\ASUS\AppData\Local\Programs\Microsoft VS Code\charts/'.
